In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
print("Libraries loaded!")

Libraries loaded!


In [4]:
# 1. Load the data, skipping the second descriptive row
income_df = pd.read_csv('data\ACSST5Y2024.S1901-Data.csv', skiprows=[1])

# 2. Keep only the columns we need: GEO_ID and Median Income
# Column S1901_C01_012E = Estimate!!Households!!Median income
income_df = income_df[['GEO_ID', 'S1901_C01_012E']].copy()

# 3. Rename for clarity
income_df.columns = ['geo_id', 'median_income']

# 4. Extract the 5-digit Zip Code from GEO_ID (e.g., '8600000US10001' -> '10001')
income_df['zip_code'] = income_df['geo_id'].str.split('US').str[-1]

# 5. Convert income to numeric (removes non-number strings like "-" or "250,000+")
income_df['median_income'] = pd.to_numeric(income_df['median_income'], errors='coerce')

# Drop rows where we don't have income data
income_df = income_df.dropna(subset=['median_income'])

print(f"Loaded income data for {len(income_df)} Zip Codes.")
income_df.head()

Loaded income data for 1617 Zip Codes.


,geo_id,median_income,zip_code
0,860Z200US06390,98125.0,06390
1,860Z200US10001,129852.0,10001
2,860Z200US10002,48386.0,10002
3,860Z200US10003,154262.0,10003
5,860Z200US10005,190233.0,10005


In [9]:
# Load the tree data (You can start with 2nd parameter nrows=10000 to test, then remove nrows later)
trees_df = pd.read_csv('data/2015_Street_Tree_Census_-_Tree_Data_20260315.csv')

# 1. Filter to only 'Alive' trees (we can't measure the health of a stump)
trees_df = trees_df[trees_df['status'] == 'Alive']

# 2. Keep only the relevant columns
trees_df = trees_df[['tree_id', 'health', 'postcode', 'spc_common']]

# 3. Ensure Zip Code (postcode) is a string to match the income data
trees_df['postcode'] = trees_df['postcode'].astype(int).astype(str)

print(f"Loaded {len(trees_df)} living trees.")
trees_df.head()

Loaded 652173 living trees.


,tree_id,health,postcode,spc_common
0,180683,Fair,11375,red maple
1,200540,Fair,11357,pin oak
2,204026,Good,11211,honeylocust
3,204337,Good,11211,honeylocust
4,189565,Good,11215,American linden


In [10]:
# 1. Count how many trees are in each Zip Code
tree_counts = trees_df.groupby('postcode')['tree_id'].count().reset_index()
tree_counts.columns = ['zip_code', 'tree_count']

# 2. Calculate "Health Score"
# We map Good=3, Fair=2, Poor=1 to turn categories into a math average
health_map = {'Good': 3, 'Fair': 2, 'Poor': 1}
trees_df['health_numeric'] = trees_df['health'].map(health_map)

avg_health = trees_df.groupby('postcode')['health_numeric'].mean().reset_index()
avg_health.columns = ['zip_code', 'avg_health_score']

# 3. Merge tree stats together
tree_stats = pd.merge(tree_counts, avg_health, on='zip_code')

# 4. FINAL MERGE: Link Tree Stats with Income!
final_df = pd.merge(tree_stats, income_df, on='zip_code')

print("Final Combined Dataset Created!")
final_df.head()

Final Combined Dataset Created!


,zip_code,tree_count,avg_health_score,geo_id,median_income
0,10001,850,2.809412,860Z200US10001,129852.0
1,10002,2158,2.714087,860Z200US10002,48386.0
2,10003,1943,2.709213,860Z200US10003,154262.0
3,10005,130,2.346154,860Z200US10005,190233.0
4,10006,48,2.875000,860Z200US10006,190170.0
